# Live trading engine

1. Know where the last run stopped
2. Know what kind of day it is (i.e. last day of the month)
3. Valuing the portfolio as it moves on normal days
4. Running the allocators live on decision days
5. Trading after decision days
6. Recording results

In [32]:
# Imports

import pandas as pd
import numpy as np
import duckdb
import pytest
from vportfolio.params import LIVE_START, CAP, LOOKBACK, RAW_FILE, TRIALS_LOG, BACKTEST, LIVE_PORTFOLIOS
from vportfolio import store, pipelines, engine, allocators, data
from vportfolio.load_data import load_benchmark
from tests import helpers

In [33]:
con = duckdb.connect() # no file name: a temporary database in memory

## Know where the last run stopped

In [34]:
# A function to compare the last date in store with the new downloaded data
# returns the dates still to process, in order
def new_days (dates, last, start = LIVE_START):
    new_dates = dates[dates >= pd.Timestamp(start)]
    if last is not None:
        new_dates = new_dates[new_dates > pd.Timestamp(last)]
    return new_dates

In [35]:
# testing
test_dates = pd.bdate_range("2026-11-25", "2026-12-04")
# 1. always true: in date order, and only dates from the download
test_result = new_days(test_dates, pd.Timestamp("2026-12-02"))
assert test_result.is_monotonic_increasing and test_result.isin(test_dates).all()
# 2. known answer: first run starts at the live start; later runs start after the last recorded day
assert list(new_days(test_dates, None)) == list(pd.bdate_range("2026-12-01", "2026-12-04"))
assert list(test_result) == [pd.Timestamp("2026-12-03"), pd.Timestamp("2026-12-04")]
assert len(new_days(test_dates, pd.Timestamp("2026-12-04"))) == 0
# 3. inputs matter: a later start drops the earlier days; a download that ends before the start gives nothing
assert new_days(test_dates, None, start = "2026-12-03")[0] == pd.Timestamp("2026-12-03")
november = pd.bdate_range("2026-11-02", "2026-11-27")
assert len(new_days(november, None)) == 0

print ("new_days passed all tests")

new_days passed all tests


## Know what kind of day it is (i.e. last day of the month)

In [36]:
# Decide if the date in question is a decision day, a trade day or a normal day
# Returns a series of labels for the given dates
def day_kind (todo, rebal_dates, dates):
    pos = dates.get_indexer(rebal_dates) # step 1: positions of the decision days
    next_pos = pos + 1 # step 2: one further 
    next_pos = next_pos[next_pos < len(dates)] # adjust for runs on decision days
    trade_days = dates[next_pos] # step 3: the dates
    kinds = pd.Series("normal", index = todo)
    kinds[todo.isin(trade_days)] = "trade"
    kinds[todo.isin(rebal_dates)] = "decision"
    return kinds

In [37]:
# testing
test_dates = pd.bdate_range("2026-10-28", "2026-12-04")
test_rebal = pd.Series(pd.to_datetime(["2026-10-30", "2026-11-30"])) # Friday, Monday
test_kinds = day_kind(test_dates, test_rebal, test_dates)

# 1. always true: one label per day to process, and only the three labels
assert test_kinds.index.equals(test_dates)
assert set(test_kinds.unique()) <= {"normal", "decision", "trade"}
# 2. known answer: month-ends are decisions; the next trading day is a trade, across a weekend too
assert test_kinds["2026-10-30"] == "decision" and test_kinds["2026-11-02"] == "trade" # Fri → Mon
assert test_kinds["2026-11-30"] == "decision" and test_kinds["2026-12-01"] == "trade"
assert (test_kinds == "normal").sum() == len(test_dates) - 4
# 3. inputs matter: a trade day is found even when its decision day is not among the days to process
assert list(day_kind(test_dates[test_dates == "2026-12-01"], test_rebal, test_dates)) == ["trade"]
# 3. inputs matter: a download ending on the decision day labels it, with no trade day yet and no error
short = test_dates[test_dates <= "2026-11-30"]
assert day_kind(short[-1:], test_rebal, short).iloc[0] == "decision"
# 3. inputs matter: no decision days, every day is normal
no_rebal = pd.Series([], dtype = "datetime64[ns]")
assert (day_kind(test_dates, no_rebal, test_dates) == "normal").all()

print ("day_kind passed all tests")

day_kind passed all tests


## Valuing the portfolio as it moves on normal days

In [38]:
# Actions required for normal days: hold, and let the weights drift with the day's returns
# Returns the portfolio's return for the day and the drifted weights at today's close
def normal_day(holdings, returns):
    held_returns = returns[holdings.index] # keep returns of held tickets
    if held_returns.isna().any(): # make sure no missing return impacts calculations
        raise ValueError(f"missing returns for held tickers: {list(held_returns[held_returns.isna()].index)}")
    grown = holdings * (held_returns +1)
    day_return = grown.sum() - 1
    new_holdings = grown / grown.sum()
    return day_return, new_holdings

In [39]:
# testing
test_holdings = pd.Series({"A": 0.5, "B": 0.5})
test_returns = pd.Series({"A": 0.10, "B": 0.0, "C": 0.30}) # C is not held
day_return, new_holdings = normal_day(test_holdings, test_returns)

# 1. always true: new weights sum to 1, same tickers as before
assert np.isclose(new_holdings.sum(), 1)
assert list(new_holdings.index) == ["A", "B"]
# 2. known answer: 50/50, A +10%, B flat → +5%; A grows to 0.55/1.05
assert np.isclose(day_return, 0.05)
assert np.isclose(new_holdings["A"], 0.55 / 1.05)
# 2. known answer: two days in a row equal the engine's buy-and-hold (1.10 × 1.10 and 0.90 → 1.055)
r2, h2 = normal_day(new_holdings, pd.Series({"A": 0.10, "B": -0.10}))
assert np.isclose((1 + day_return) * (1 + r2), 0.5 * 1.21 + 0.5 * 0.90)
# 3. inputs matter: a ticker not held does not change the result
assert np.isclose(normal_day(test_holdings, test_returns.drop("C"))[0], day_return)
# 3. inputs matter: a held ticker with no return stops with an error
with pytest.raises(ValueError):
    normal_day(test_holdings, pd.Series({"A": 0.10, "B": np.nan}))

print ("normal_day passed all tests")

normal_day passed all tests


## Running the allocators live on decision days

In [40]:
# A wrapper running the allocator live on decision days
# Returns a series with ticker and target weight
def decide_targets(allocator, processed, day, cap = CAP, lookback = LOOKBACK):
    return engine.new_target(allocator, processed["returns"], processed["eligible"], day, cap, lookback, processed["returns"].columns)

In [41]:
# access processed data for testing
raw = pd.read_parquet(RAW_FILE)
processed = pipelines.data_pipeline(raw)

In [42]:
# testing
bench = load_benchmark()
test_day = processed["rebal_dates"].iloc[-1]

nco = decide_targets(allocators.nco, processed, test_day)
ew = decide_targets(allocators.equal_weights, processed, test_day)
sf = decide_targets(allocators.sixty_forty, bench, test_day, cap = None)

# 1. always true: every portfolio is fully invested, with no negative weights
for name, w in [("nco", nco), ("ew", ew), ("sf", sf)]:
    assert np.isclose(w.sum(), 1) and (w >= 0).all(), (name, w.sum(), w.min())
# 2. known answers: NCO respects the cap; 60/40 is 60% SPY and 40% AGG
assert nco.max() <= CAP + 1e-9
assert np.isclose(sf["SPY"], 0.6) and np.isclose(sf["AGG"], 0.4)
# 2. known answer: identical to calling the engine directly, as the backtest does
assert nco.equals(engine.new_target(allocators.nco, processed["returns"], processed["eligible"],
                                    test_day, CAP, LOOKBACK, processed["returns"].columns))
# 3. inputs matter: tickers not eligible get nothing; different allocators give different weights
not_eligible = ~processed["eligible"].loc[test_day]
assert (nco[not_eligible] == 0).all()
assert not nco.equals(ew)

print ("decide_targets passed all tests")

decide_targets passed all tests


### Data calculation for the 60/40 allocator case

This is needed because AGY is not in the universe of data but only in the benchmark

In [43]:
# Build the data set the 60/40 runs on, from a raw download
# Returns a dictionary with close, open, returns and eligible
def benchmark_data(raw, tickers = ["SPY", "AGG"]):
    close = raw["Close"][tickers]
    open_ = raw["Open"][tickers]
    returns = close.pct_change(fill_method=None)
    eligible = close.notna()
    return {"close": close, "open": open_, "returns": returns, "eligible": eligible}

In [44]:
# testing
fake = helpers.make_fake_raw("2022-03-15")
b = benchmark_data(fake)
# 1. always true: four tables, each with SPY and AGG only, on the raw data's dates
assert set(b.keys()) == {"close", "open", "returns", "eligible"}
for key, table in b.items():
    assert table.columns.tolist() == ["SPY", "AGG"] and table.index.equals(fake.index), key
# 2. known answer: same values as the raw data; returns are the closes' daily change
pd.testing.assert_frame_equal(b["close"], fake["Close"][["SPY", "AGG"]])
assert np.isclose(b["returns"]["SPY"].iloc[1], fake["Close"]["SPY"].iloc[1] / fake["Close"]["SPY"].iloc[0] - 1)
# 3. inputs matter: other tickers give other columns
assert benchmark_data(fake, ["SPY"])["close"].columns.tolist() == ["SPY"]

print ("benchmark_data passed all tests")

benchmark_data passed all tests


## Trading after decision days

In [45]:
# Simulate trading on a trading day
# Returns a series with portfolio's daily return, new holdings and trades
def trade_day(holdings, targets, overnight, intraday, cost_bp):
    # 1. overnight: the holdings move to today's open (nothing held on the first day)
    if holdings.empty:
        overnight_return = 0.0
        weights_at_open = pd.Series(0.0, index = targets.index)
    else:
        overnight_return, weights_at_open = normal_day(holdings, overnight)
    # line up both on every ticker that is either held or targeted, 0 where absent
    tickers = targets.index.union(weights_at_open.index)
    before = weights_at_open.reindex(tickers, fill_value = 0.0)
    after = targets.reindex(tickers, fill_value = 0.0)
    # 2. trade at the open: same cost rule as the backtest
    costs = engine.trading_costs(after, before, cost_bp.reindex(tickers))
    # 3. during the day: the new weights move to today's close (only tickers actually held)
    intraday_return, new_holdings = normal_day(after[after > 0], intraday)
    # 4. the day's return, chaining the three moves
    day_return = (1 + overnight_return) * (1 - costs["cost"]) * (1 + intraday_return) - 1
    # 5. the trades: every ticker whose weight changed
    changed = (after - before).abs() > 1e-12
    trades = pd.DataFrame({"ticker": tickers[changed],
                           "weight_before": before[changed].values,
                           "weight_after": after[changed].values,
                           "cost": ((after - before).abs() * cost_bp.reindex(tickers) / 10000)[changed].values})
    return day_return, new_holdings, trades

In [46]:
# testing
h = pd.Series({"A": 0.5, "B": 0.5})
t = pd.Series({"A": 0.6, "B": 0.4})
on = pd.Series({"A": 0.02, "B": 0.0})
intra = pd.Series({"A": 0.01, "B": -0.01})
bp = pd.Series({"A": 10.0, "B": 10.0})
r, new_h, trades = trade_day(h, t, on, intra, bp)

# 1. always true: weights at the close sum to 1; the trades end exactly at the targets; costs are not negative
assert np.isclose(new_h.sum(), 1)
assert np.allclose(trades.set_index("ticker")["weight_after"], t)
assert (trades["cost"] >= 0).all()
# 2. known answer: overnight +1%, then cost on 0.19 of turnover at 10 bp, then +0.2% during the day
open_w = pd.Series({"A": 0.51, "B": 0.50}) / 1.01
cost = (t - open_w).abs().sum() * 0.001
assert np.isclose(r, 1.01 * (1 - cost) * 1.002 - 1)
assert np.isclose(trades["cost"].sum(), cost)
# 2. known answer: the first day (nothing held) matches the backtest engine's first trade day
day1, day2 = pd.Timestamp("2026-12-01"), pd.Timestamp("2026-12-02")
open_p = pd.DataFrame({"A": [100.0, 100.0], "B": [100.0, 100.0]}, index = [day1, day2])
close_p = pd.DataFrame({"A": [101.0, 101.0], "B": [99.0, 99.0]}, index = [day1, day2])
r0, h0, trades0 = trade_day(pd.Series(dtype = float), t, on, intra, bp)
engine_cost = engine.trading_costs(t, pd.Series(0.0, index = t.index), bp)["cost"]
engine_nav = engine.holding_nav(1 - engine_cost, t, close_p, open_p, day1, day2).iloc[0]
assert np.isclose(1 + r0, engine_nav)
assert len(trades0) == 2 and (trades0["weight_before"] == 0).all()
# 3. inputs matter: no overnight move and targets equal to holdings → no trades, no cost
r_same, _, trades_same = trade_day(h, h, pd.Series({"A": 0.0, "B": 0.0}), intra, bp)
assert trades_same.empty and np.isclose(r_same, 0.0)
# 3. inputs matter: higher costs give a lower return
r_dear, _, _ = trade_day(h, t, on, intra, bp * 10)
assert r_dear < r
# 3. inputs matter: a held ticker that is no longer targeted is sold, and it is not lost
h_c = pd.Series({"A": 0.4, "B": 0.4, "C": 0.2})
r_c, h_c_new, trades_c = trade_day(h_c, t, pd.Series({"A": 0.0, "B": 0.0, "C": 0.0}), intra, bp)
sold = trades_c.set_index("ticker").loc["C"]
assert sold["weight_before"] == 0.2 and sold["weight_after"] == 0.0
assert "C" not in h_c_new.index

print ("trade_day passed all tests")

trade_day passed all tests


# Live trading pipeline

We will first run a trial and then build it

In [47]:
con = duckdb.connect()

In [48]:
# start by creating the empty tables
store.create_tables(con)

In [49]:
# get last date
last = store.last_date(con)

In [50]:
# get the portfolio navs
navs = store.last_nav(con).set_index("portfolio")["nav"] # set_index here turns tables into a lookup

In [51]:
# get the weights series
holdings = {}
for name, rows in store.last_holdings(con).groupby("portfolio"): #groupby splits the holdings table into one piece per portfolio
    holdings[name] = rows.set_index("ticker")["weight"]

In [52]:
# select the dates from processed data
dates = processed["returns"].index

In [53]:
# make a new start date from september 1st
todo = new_days(dates, last, start = "2026-09-01")

A start date of September 1st should guve ys two trade days (1st of September and 1st of October), one decision day (30th of September) and the rest should be all normal days

In [54]:
# get the days types
kinds = day_kind(todo,processed["rebal_dates"], dates)

In [55]:
kinds.value_counts()

normal      34
trade        2
decision     1
Name: count, dtype: int64

Before building the complete loop, we look at the first day alone for NCO only and look at the result

In [56]:
# helper function to build the three kinds of returns a day needs
# returns the three values fo overnight (yesterday's close and today's open), intraday (today's open and close) and full (yesterday's and today's close)
def day_returns(data, day):
    close, open_ = data["close"], data["open"]
    previous_close = close.iloc[close.index.get_loc(day) - 1]
    overnight = open_.loc[day] / previous_close - 1
    intraday = close.loc[day] / open_.loc[day] - 1
    full = data["returns"].loc[day]
    return overnight, intraday, full

In [57]:
# day_returns
def test_day_returns():
    days = pd.bdate_range("2026-12-01", periods = 3)
    close = pd.DataFrame({"A": [100.0, 110.0, 99.0]}, index = days)
    open_ = pd.DataFrame({"A": [100.0, 105.0, 110.0]}, index = days)
    prices = {"close": close, "open": open_, "returns": close.pct_change(fill_method = None)}
    overnight, intraday, full = day_returns(prices, days[1])
    # 1. always true: overnight and intraday chain into the full day: (1 + o) × (1 + i) = 1 + full
    assert np.isclose((1 + overnight["A"]) * (1 + intraday["A"]), 1 + full["A"])
    # 2. known answer: close 100 → open 105 is +5%; open 105 → close 110 is +4.76%; close to close +10%
    assert np.isclose(overnight["A"], 0.05)
    assert np.isclose(intraday["A"], 110 / 105 - 1)
    assert np.isclose(full["A"], 0.10)
    # 3. inputs matter: the next day uses its own previous close (110), not the first day's
    overnight_3, _, _ = day_returns(prices, days[2])
    assert np.isclose(overnight_3["A"], 0.0)          # close 110 → open 110

In [58]:
test_day_returns()
print ("day_returns passed all tests")

day_returns passed all tests


In [59]:
# get the data and put them in a dictionary
bench = benchmark_data(raw)
portfolio_data = {"nco": processed, "equal_weights": processed, "sixty_forty": bench}

In [60]:
# set the day to 1st of September
day = todo[0] 

In [61]:
# set the nco for the run
name = "nco"
settings = LIVE_PORTFOLIOS[name]
allocator = getattr(allocators, settings["allocator"]) # turns the name "nco" into the function

In [62]:
# set the decision date
rebal = processed["rebal_dates"]
decision = rebal[rebal < day].iloc[-1] # the month-end before the trade day

In [63]:
# run the wrapper function with the params just defined (we only select NCO)
targets = decide_targets(allocator, portfolio_data[name], decision, settings["cap"], settings["lookback"])

In [64]:
# run the helper function to get the returns
overnight, intraday, full = day_returns(portfolio_data[name], day)

In [65]:
#  finally we get the daily return, new holdings and trades
r, new_holdings, trades = trade_day(pd.Series(dtype = float), targets, overnight, intraday, processed["cost_bp"])

- The decision date should be the August 31st, the month end before September 1st
- We should have a trade for each ticker since we start at 0 weight
- Cap should be respected

In [66]:
print(decision, round(r, 5), round(1 * (1 + r), 5))
print(new_holdings.round(3))
trades

2026-08-31 00:00:00 -0.00072 0.99928
Ticker
QQQ     0.000
EFA     0.032
EEM     0.008
IEF     0.061
SHY     0.200
TIP     0.200
LQD     0.000
HYG     0.200
EMB     0.028
EMLC    0.063
DBC     0.012
XLE     0.000
VNQ     0.006
UUP     0.150
FXY     0.007
FXA     0.006
FXF     0.027
dtype: float64


,ticker,weight_before,weight_after,cost
0,EFA,0.0,0.032515,0.000016
1,EEM,0.0,0.007671,0.000002
2,IEF,0.0,0.060881,0.000012
3,SHY,0.0,0.200000,0.000040
4,TIP,0.0,0.200000,0.000040
5,HYG,0.0,0.200000,0.000100
6,EMB,0.0,0.027865,0.000028
7,EMLC,0.0,0.062905,0.000063
8,DBC,0.0,0.012212,0.000012
9,VNQ,0.0,0.005660,0.000003


Now we run a trial with 60/40

In [67]:
name = "sixty_forty"
settings = LIVE_PORTFOLIOS[name]
allocator = getattr(allocators, settings["allocator"])
pdata = portfolio_data[name]

rebal = processed["rebal_dates"]
decision = rebal[rebal < day].iloc[-1]

targets = decide_targets(allocator, pdata, decision, settings["cap"], settings["lookback"])
overnight, intraday, full = day_returns(pdata, day)
r, new_holdings, trades = trade_day(pd.Series(dtype = float), targets, overnight, intraday, processed["cost_bp"])

In [68]:
print(decision, round(r, 5), round(1 * (1 + r), 5))
print(new_holdings.round(3))
trades

2026-08-31 00:00:00 -0.00084 0.99916
Ticker
SPY    0.6
AGG    0.4
dtype: float64


,ticker,weight_before,weight_after,cost
0,SPY,0.0,0.6,0.00012
1,AGG,0.0,0.4,0.00008


Everything checks out so now we can run the trial on all days and all allocators

In [69]:
# We get the last NAV and prepare the series for storage
navs = store.last_nav(con).set_index("portfolio")["nav"].to_dict() # {} on the first run
nav_rows, holding_rows, target_rows, trade_rows = [], [], [], []

In [70]:
# fresh in-memory store: the same start as the very first live run
con = duckdb.connect()
store.create_tables(con)

# where each portfolio stopped (all empty on the first run)
navs = store.last_nav(con).set_index("portfolio")["nav"].to_dict()
holdings = {name: rows.set_index("ticker")["weight"]
            for name, rows in store.last_holdings(con).groupby("portfolio")}
nav_rows, holding_rows, target_rows, trade_rows = [], [], [], []

for day in todo:
    kind = kinds[day]
    for name, settings in LIVE_PORTFOLIOS.items():
        pdata = portfolio_data[name]
        held = holdings.get(name, pd.Series(dtype = float)) # nothing held yet = empty
        nav = navs.get(name, 1.0) # no NAV yet = start at 1
        overnight, intraday, full = day_returns(pdata, day)
        # if the decision is a trade day, run decide_targets and trade_day
        if kind == "trade":
            allocator = getattr(allocators, settings["allocator"])
            decision = rebal[rebal < day].iloc[-1]
            targets = decide_targets(allocator, pdata, decision, settings["cap"], settings["lookback"])
            r, new_held, trades = trade_day(held, targets, overnight, intraday, processed["cost_bp"])
            target_rows.append(pd.DataFrame({"decision_date": decision, "portfolio": name,
                                             "ticker": targets.index, "weight": targets.values}))
            trade_rows.append(trades.assign(trade_date = day, portfolio = name)) # assign adds the two columns
        # otherwise we run normal_day and just get portfolio's return and drifted weights
        else:
            if held.empty:
                # protects us from a start date that is not a trading day. else it would record a flat NAV
                raise ValueError(f"{name} holds nothing on {day.date()}: the first live day must be a trade day")
            r, new_held = normal_day(held, full)
        # then we calculate the NAV
        nav = nav * (1 + r)
        navs[name], holdings[name] = nav, new_held # puts today's results in to memory so- next day starts there
        nav_rows.append({"date": day, "portfolio": name, "nav": nav, "daily_return": r})
        holding_rows.append(pd.DataFrame({"date": day, "portfolio": name,
                                          "ticker": new_held.index, "weight": new_held.values}))
# and make it a dataframe
nav_table = pd.DataFrame(nav_rows)

In [71]:
nav_table.pivot(index = "date", columns = "portfolio", values = "nav").tail() # pivot makes it easier to read

portfolio,equal_weights,nco,sixty_forty
date,,,
2026-10-16,0.988230,0.992081,0.997597
2026-10-19,0.986723,0.991111,0.998973
2026-10-20,0.989649,0.991690,1.001651
2026-10-21,0.991124,0.992382,1.004925
2026-10-22,0.989663,0.994910,1.001277


Data look relatively good:
- All moved by less than 1.2% over the seven weeks; normal for diversified portfolios in a calm period
- 60/40 is the most volatile of the three, as expected with 60% in equities

Before moving on we should cross check with the results of the backtest engine, which also covers the period we are simulating now

In [72]:
# Set the lookup range
period = rebal[(rebal >= "2026-08-31") & (rebal <= "2026-09-30")].reset_index(drop = True)

In [73]:
for name, settings in LIVE_PORTFOLIOS.items():
    # iterates over the allocators in LIVE_PORTFOLIOS
    pdata = portfolio_data[name]
    # run the allocator functions with given settings
    result = engine.backtest_engine(getattr(allocators, settings["allocator"]),
                                    pdata["close"], pdata["open"], pdata["returns"],
                                    pdata["eligible"], period, processed["cost_bp"],
                                    settings["cap"], settings["lookback"])
    # gets the NAV from backtest results
    backtest = result["NAV"]
    # gets the NAV from the live run
    live = nav_table[nav_table["portfolio"] == name].set_index("date")["nav"]
    # joins them in a table
    both = pd.concat({"backtest": backtest, "live": live}, axis = 1).dropna()
    print(name, len(both), "days, largest difference:", (both["backtest"] - both["live"]).abs().max())

nco 21 days, largest difference: 4.440892098500626e-16
equal_weights 21 days, largest difference: 3.3306690738754696e-16
sixty_forty 21 days, largest difference: 3.3306690738754696e-16


The result is the intended one; negligible differences between the trial live run and the backtest results. Now we build them into a pipeline

In [74]:
# Process every trading day not yet recorded, for every live portfolio
# returns the days that were processed
def daily_step(con, processed, bench, portfolios = LIVE_PORTFOLIOS, start = LIVE_START):
    # 1. where each portfolio stopped
    last = store.last_date(con) # get last date
    navs = store.last_nav(con).set_index("portfolio")["nav"].to_dict() # get last nav
    holdings = {name: rows.set_index("ticker")["weight"] # gets the weights into a dictionary
                for name, rows in store.last_holdings(con).groupby("portfolio")}
    # 2. which days are new, and what kind each one is
    dates = processed["returns"].index
    rebal = processed["rebal_dates"]
    todo = new_days(dates, last, start)
    kinds = day_kind(todo, rebal, dates)
    portfolio_data = {}
    for name, settings in portfolios.items():
        if settings["allocator"] == "sixty_forty":
            portfolio_data[name] = bench # the 60/40 runs on SPY/AGG data
        else:
            portfolio_data[name] = processed # everyone else runs on the universe
    # here we start he loop of actions
    for day in todo:
        nav_rows, holding_rows, target_rows, trade_rows = [], [], [], []
        # 3. every portfolio on this day
        for name, settings in portfolios.items():
            pdata = portfolio_data[name]
            held = holdings.get(name, pd.Series(dtype = float))
            nav = navs.get(name, 1.0)
            overnight, intraday, full = day_returns(pdata, day)
            if kinds[day] == "trade":
                allocator = getattr(allocators, settings["allocator"])
                decision = rebal[rebal < day].iloc[-1]
                targets = decide_targets(allocator, pdata, decision, settings["cap"], settings["lookback"])
                r, new_held, trades = trade_day(held, targets, overnight, intraday, processed["cost_bp"])
                target_rows.append(pd.DataFrame({"decision_date": decision, "portfolio": name,
                                                 "ticker": targets.index, "weight": targets.values}))
                trade_rows.append(trades.assign(trade_date = day, portfolio = name))
            else:
                if held.empty:
                    raise ValueError(f"{name} holds nothing on {day.date()}: the first live day must be a trade day")
                r, new_held = normal_day(held, full)
            nav = nav * (1 + r)
            navs[name], holdings[name] = nav, new_held
            nav_rows.append({"date": day, "portfolio": name, "nav": nav, "daily_return": r})
            holding_rows.append(pd.DataFrame({"date": day, "portfolio": name,
                                              "ticker": new_held.index, "weight": new_held.values}))

        # 4. the returns used that day, for every ticker with a price (SPY appears in both sets, kept once)
        used = pd.concat([processed["returns"].loc[day], bench["returns"].loc[day]])
        used = used[~used.index.duplicated()].dropna()
        return_rows = pd.DataFrame({"date": day, "ticker": used.index, "daily_return": used.values})
        # 5. write the whole day at once: all tables, or none
        con.begin()
        try:
            store.append_rows(con, "nav", pd.DataFrame(nav_rows))
            store.append_rows(con, "holdings", pd.concat(holding_rows))
            store.append_rows(con, "asset_returns", return_rows)
            if target_rows:
                store.append_rows(con, "targets", pd.concat(target_rows))
                store.append_rows(con, "trades", pd.concat(trade_rows))
            con.commit()
        except Exception:
            con.rollback()
            raise
    return todo

In [75]:
# fake universe and benchmark data, and two cheap test portfolios
def make_fake_live():
    # fake universe and benchmark data, and two cheap test portfolios
    raw = helpers.make_fake_raw("2022-03-15")
    processed = pipelines.data_pipeline(raw)
    bench = benchmark_data(raw)
    portfolios = {"ew": {"allocator": "equal_weights", "cap": 0.2,  "lookback": 504},
                  "sf": {"allocator": "sixty_forty",   "cap": None, "lookback": 1}}
    return raw, processed, bench, portfolios

In [76]:
def test_daily_step():
    raw, processed, bench, portfolios = make_fake_live()
    start = "2021-06-01" # the trading day after the May 2021 month-end
    con = helpers.make_store()
    done = daily_step(con, processed, bench, portfolios, start)
    navs = con.execute("SELECT * FROM nav").df()

    # 1. always true: one NAV per day and portfolio; weights sum to 1 every day; a second run adds nothing
    assert len(navs) == len(done) * 2
    sums = con.execute("SELECT date, portfolio, sum(weight) AS total FROM holdings GROUP BY date, portfolio").df()
    assert np.allclose(sums["total"], 1)
    assert len(daily_step(con, processed, bench, portfolios, start)) == 0
    assert con.execute("SELECT count(*) FROM nav").fetchone()[0] == len(navs)

    # 2. known answer: the first day is a trade day; the 60/40 trades SPY and AGG and targets 0.6 / 0.4
    first_trades = con.execute("SELECT * FROM trades WHERE trade_date = '2021-06-01' AND portfolio = 'sf'").df()
    assert set(first_trades["ticker"]) == {"SPY", "AGG"}
    sf_targets = con.execute("SELECT ticker, weight FROM targets WHERE portfolio = 'sf'").df()
    assert np.allclose(sf_targets.groupby("ticker")["weight"].first().loc[["SPY", "AGG"]], [0.6, 0.4])

    # 3. inputs matter: running in two parts gives the same NAVs as one run (the daily case)
    con_parts = helpers.make_store()
    short_raw = raw.loc[:"2021-09-15"]
    daily_step(con_parts, pipelines.data_pipeline(short_raw), benchmark_data(short_raw), portfolios, start)
    daily_step(con_parts, processed, bench, portfolios, start)
    parts = con_parts.execute("SELECT * FROM nav ORDER BY date, portfolio").df()
    pd.testing.assert_frame_equal(parts, navs.sort_values(["date", "portfolio"]).reset_index(drop = True))

    # 3. inputs matter: a start that is not a trade day stops with an error and writes nothing
    con_bad = helpers.make_store()
    with pytest.raises(ValueError):
        daily_step(con_bad, processed, bench, portfolios, "2021-06-02")
    assert con_bad.execute("SELECT count(*) FROM nav").fetchone()[0] == 0

In [77]:
test_daily_step()
print("daily_step passed all tests")

daily_step passed all tests
